# Reasoning Model Validation

Use this notebook to inspect chatsnack's current reasoning capability table, capture local warnings, and optionally try live calls against real models.


In [ ]:
import os
import warnings
from pprint import pprint

from chatsnack import Chat
from chatsnack.chat.mixin_params import ChatParams

MODELS = [
    'gpt-5',
    'gpt-5.1',
    'gpt-5.4',
    'gpt-5.4-mini',
    'gpt-5.4-nano',
    'o3-mini',
    'o4-mini',
]

REASONING_CASES = [
    {'effort': None, 'summary': None},
    {'effort': 'low', 'summary': None},
    {'effort': 'medium', 'summary': 'auto'},
    {'effort': 'high', 'summary': 'detailed'},
    {'effort': 'xhigh', 'summary': 'concise'},
]

RUN_LIVE = bool(os.environ.get('OPENAI_API_KEY'))


In [ ]:
def inspect_reasoning_case(model, effort=None, summary=None):
    responses = {}
    authored = {k: v for k, v in {'effort': effort, 'summary': summary}.items() if v is not None}
    if authored:
        responses['reasoning'] = authored

    params = ChatParams(model=model, runtime='responses', responses=responses or None)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        request_opts = params._get_responses_api_options()

    return {
        'model': model,
        'authored': authored or None,
        'known_capabilities': params._get_reasoning_capabilities(),
        'request_reasoning': request_opts.get('reasoning'),
        'warnings': [str(w.message) for w in caught],
    }

def try_live_reasoning_case(model, effort=None, summary=None, prompt='Give one sentence about why snacks are fun.'):
    if not RUN_LIVE:
        return {'model': model, 'skipped': 'Set OPENAI_API_KEY to run live checks.'}

    chat = Chat(prompt, model=model)
    chat.reasoning.effort = effort
    chat.reasoning.summary = summary
    try:
        return {
            'model': model,
            'effort': effort,
            'summary': summary,
            'response': chat.ask('Keep it short.'),
        }
    except Exception as exc:
        return {
            'model': model,
            'effort': effort,
            'summary': summary,
            'error': type(exc).__name__,
            'message': str(exc),
        }


In [ ]:
local_results = [inspect_reasoning_case(model, **case) for model in MODELS for case in REASONING_CASES]
for row in local_results:
    pprint(row)
    print('-' * 80)


In [ ]:
# Edit these and re-run to probe a specific model live.
LIVE_MODEL = 'gpt-5.4'
LIVE_EFFORT = 'xhigh'
LIVE_SUMMARY = 'auto'

pprint(try_live_reasoning_case(LIVE_MODEL, effort=LIVE_EFFORT, summary=LIVE_SUMMARY))
